# Datamine Studio RM - Holes3D Scripting Tutorial

This Jupyter Notebook demonstrates how to use the `dmstudio` Python package to automate Datamine processes via COM. 

The workflow is based on the **Holes3D Scripting Tutorial** from the official Datamine help documentation. It performs drillhole de-surveying (calculating 3D spatial coordinates) by combining and processing collars, assays, lithology, and survey data files.

## Workflow Overview
1. **Sort**: Sort all input files (`collars`, `assays`, `lithology`, `surveys`) on their key fields using `MGSORT`.
2. **Merge**: Merge the assay and lithology interval files using `HOLMER`.
3. **Join**: Relational join the collars file with the merged samples file using `JOIN`.
4. **De-survey**: Calculate 3D coordinates for all samples using `DESURV`.
5. **Display**: Export the final data as a CSV and view it using `pandas`.

In [1]:
# Step 1: Initialize connection to Datamine Studio RM
from dmstudio import dmcommands, initialize, special
import pandas as pd

# Connect to Studio RM
dm = dmcommands.init(version='StudioRM')
oScript = initialize.studio('StudioRM')
print(f"Connected to: {oScript.Caption}")
# Verify that the active project matches this folder (case-insensitive) to prevent writing files to the wrong place
import os
notebook_folder = os.path.normpath(os.path.dirname(os.path.abspath('__file__'))).lower()
active_folder = os.path.normpath(oScript.ActiveProject.Folder).lower()
if active_folder != notebook_folder:
    raise RuntimeError(
        f"Active Datamine Project ({active_folder}) does not match this notebook's folder ({notebook_folder}).\n"
        "Please open the 'Project.rmproj' in this folder inside Datamine Studio RM first!"
    )
print("Active project verified successfully.")

Connected to: Studio RM 3.1.390.0 - Project.rmproj * - [3D]


In [2]:
# Step 2: Import local file mappings
# We import dmdir which contains variables pointing to the copied files in the project root.
import dmdir as f

print("Registered source files:")
print(f"  Collars: {f.__vb_collars_}")
print(f"  Assays: {f.__vb_assays_}")
print(f"  Lithology: {f.__vb_lithology_}")
print(f"  Surveys: {f.__vb_surveys_}")

Registered source files:
  Collars: _vb_collars
  Assays: _vb_assays
  Lithology: _vb_lithology
  Surveys: _vb_surveys


In [3]:
# Step 3: Sort the input tables on their respective key fields
print("Sorting collars by BHID...")
dm.mgsort(in_i=f.__vb_collars_, out_o='tempcollar', keys_f=['BHID'])

print("Sorting assays by BHID and FROM...")
dm.mgsort(in_i=f.__vb_assays_, out_o='tempassays', keys_f=['BHID', 'FROM'])

print("Sorting lithology by BHID and FROM...")
dm.mgsort(in_i=f.__vb_lithology_, out_o='templith', keys_f=['BHID', 'FROM'])

print("Sorting surveys by BHID and AT...")
dm.mgsort(in_i=f.__vb_surveys_, out_o='tempsurvey', keys_f=['BHID', 'AT'])

print("Sorting completed.")

Sorting collars by BHID...
Sorting assays by BHID and FROM...
Sorting lithology by BHID and FROM...
Sorting surveys by BHID and AT...
Sorting completed.


In [4]:
# Step 4: Merge assays and lithology tables using HOLMER
print("Merging assays and lithology (HOLMER)...")
dm.holmer(
    inmods_i=['tempassays', 'templith'],
    out_o='temp1',
    bhid_f='BHID',
    from_f='FROM',
    to_f='TO'
)
print("Merge completed.")

Merging assays and lithology (HOLMER)...
Merge completed.


In [5]:
# Step 5: Join collars with merged sample intervals using JOIN
print("Relational joining collars and samples (JOIN)...")
dm.join(
    inmods_i=['tempcollar', 'temp1'],
    out_o='temp2',
    keys_f=['BHID'],
    subsetr_p=1,
    subsetf_p=0,
    cartjoin_p=0,
    print_p=0
)
print("Join completed.")

Relational joining collars and samples (JOIN)...
Join completed.


In [6]:
# Step 6: De-survey the drillholes using DESURV to get 3D coordinates
print("De-surveying drillholes (DESURV)...")
dm.desurv(
    inmods_i=['temp2', 'tempsurvey'],
    out_o='dholes',
    survsmth_p=1,
    print_p=0
)
print("De-survey completed.")

De-surveying drillholes (DESURV)...
De-survey completed.


## Step 7: View the De-surveyed Data Definition

We inspect the spatial coordinate fields (`X`, `Y`, `Z`) created by `DESURV`.

In [7]:
# View final de-surveyed data fields
dm.ddlist(in_i='dholes')

## Step 8: Downhole Compositing (COMPDH)

We composite the drillhole assay samples into constant 2.0-meter intervals down each hole. This standardizes the sample support size before grade estimation.

In [8]:
print("Compositing drillholes (COMPDH)...")
dm.compdh(
    in_i='dholes',
    out_o='comp_holes',
    interval_p=2.0
)
print("Compositing completed.")

Compositing drillholes (COMPDH)...
Compositing completed.


## Step 9: Visualization

We visualize the composited drillhole grades both inside the Datamine 3D window and as a 3D scatter plot directly within JupyterLab.

In [10]:
comp_holes_temp_path

'D:\\Active\\dmstudio\\tutorials\\comp_holes_temp.dmx'

In [9]:
# 1. Copy the file to a temporary copy to bypass Datamine's write lock
import shutil
import os
import json
import pandas as pd
project_dir = oScript.ActiveProject.Folder
comp_holes_path = os.path.join(project_dir, 'comp_holes.dmx')
comp_holes_temp_path = os.path.join(project_dir, 'comp_holes_temp.dmx')
shutil.copy(comp_holes_path, comp_holes_temp_path)

# 2. Load the copied data into pandas using the agent utility
from dmstudio import agent
df = agent.read_datamine(comp_holes_temp_path)
print(f"Loaded {len(df)} records from {comp_holes_path}.")

# 3. Clean up the temporary file
if os.path.exists(comp_holes_temp_path):
    os.remove(comp_holes_temp_path)

# 4. Load the composited file into the active Datamine project 3D view
oScript.ActiveProject.Data.LoadFile(comp_holes_path)
print("Loaded comp_holes.dmx into Datamine active project window.")

# 5. Clean grades and coordinates (convert to numeric and filter absent values)
df['AU'] = pd.to_numeric(df['AU'], errors='coerce')
df.loc[df['AU'] < 0, 'AU'] = None  # Filter out absent value sentinel (-1e30)
df['X'] = pd.to_numeric(df['X'], errors='coerce')
df['Y'] = pd.to_numeric(df['Y'], errors='coerce')
df['Z'] = pd.to_numeric(df['Z'], errors='coerce')

# 6. Print gold grade (AU) summary statistics
print("\nGold (AU) grade summary statistics (excluding absent values):")
print(df['AU'].describe())

# 7. Generate interactive 3D WebGL HTML visualization using Plotly
df_clean = df.dropna(subset=['X', 'Y', 'Z', 'AU'])
points = []
for idx, row in df_clean.iterrows():
    points.append({
        'x': float(row['X']),
        'y': float(row['Y']),
        'z': float(row['Z']),
        'au': float(row['AU']),
        'bhid': str(row['BHID']),
        'from': float(row['FROM']),
        'to': float(row['TO'])
    })

html_content = f"""
<!DOCTYPE html>
<html>
<head>
    <title>3D Drillhole Grade Visualization</title>
    <script src=\"https://cdn.plot.ly/plotly-2.24.1.min.js\"></script>
    <style>
        body, html {{ margin: 0; padding: 0; width: 100%; height: 100%; overflow: hidden; font-family: sans-serif; }}
        #plot {{ width: 100%; height: 100%; }}
        .info-panel {{
            position: absolute; top: 10px; left: 10px; background: rgba(255,255,255,0.9);
            padding: 15px; border-radius: 8px; box-shadow: 0 4px 6px rgba(0,0,0,0.1);
            max-width: 300px; z-index: 100;
        }}
        h3 {{ margin-top: 0; color: #333; }}
        p {{ margin: 5px 0; font-size: 14px; color: #666; }}
    </style>
</head>
<body>
    <div class=\"info-panel\">
        <h3>3D Drillholes</h3>
        <p><b>Dataset:</b> comp_holes.dmx</p>
        <p><b>Intervals:</b> 2m Composites</p>
        <p><b>Variable:</b> AU Grade (g/t)</p>
        <p>Use left click + drag to rotate, right click + drag to pan, and scroll to zoom.</p>
    </div>
    <div id=\"plot\"></div>
    <script>
        const data = {json.dumps(points)};
        const x = data.map(p => p.x);
        const y = data.map(p => p.y);
        const z = data.map(p => p.z);
        const color = data.map(p => p.au);
        const text = data.map(p => `Hole: ${{p.bhid}}<br>From: ${{p.from}}m<br>To: ${{p.to}}m<br>AU: ${{p.au}} g/t`);

        const trace = {{
            x: x, y: y, z: z,
            mode: 'markers',
            marker: {{
                size: 3,
                color: color,
                colorscale: 'YlOrRd',
                colorbar: {{ title: 'AU (g/t)' }},
                opacity: 0.8
            }},
            text: text,
            hoverinfo: 'text',
            type: 'scatter3d'
        }};

        const layout = {{
            margin: {{ l: 0, r: 0, b: 0, t: 0 }},
            scene: {{
                xaxis: {{ title: 'Easting (X)' }},
                yaxis: {{ title: 'Northing (Y)' }},
                zaxis: {{ title: 'Elevation (Z)' }}
            }}
        }};

        Plotly.newPlot('plot', [trace], layout);
    </script>
</body>
</html>
"""

html_path = os.path.join(project_dir, 'drillhole_visualization.html')
with open(html_path, 'w', encoding='utf-8') as f:
    f.write(html_content)
print(f"Saved interactive 3D WebGL visualization to: {html_path}")

# 8. Display inline in Jupyter using IFrame
from IPython.display import IFrame
IFrame(src='drillhole_visualization.html', width='100%', height=600)


Loaded 3492 records from D:\Active\dmstudio\tutorials\comp_holes.dmx.
Loaded comp_holes.dmx into Datamine active project window.

Gold (AU) grade summary statistics (excluding absent values):
count    486.000000
mean       1.731336
std        2.603747
min        0.020000
25%        0.654812
50%        1.095823
75%        1.779407
max       28.415495
Name: AU, dtype: float64
Saved interactive 3D WebGL visualization to: D:\Active\dmstudio\tutorials\drillhole_visualization.html
